# Kiểm chứng mô hình sinh tải toán học

Notebook trả lời câu hỏi: **tải được sinh ra có đúng mô hình toán đã khai báo không?**

1. **Offline** (không cần cụm): sinh trace bằng chính `arrival_models.py` mà driver dùng, rồi kiểm định thống kê từng mô hình.
2. **Trên dữ liệu thật** (khi `results/` có run): so sánh tải **thiết kế** (`designed_rate.csv`) với tải **thực nhận** (span `server` của frontend), và kiểm tra driver có phải bỏ arrival nào không.

| Mô hình | Công thức | Kiểm định |
|---|---|---|
| NHPP | $\lambda(t)=\lambda_0 + A_1\sin(2\pi t/T+\varphi) + A_2\sin(4\pi t/T+2\varphi)$, sinh bằng thinning | Số lần đến mỗi phút ≈ $\int\lambda(t)\,dt$ |
| MMPP | Xích Markov thời gian liên tục, thời gian lưu trú $\sim \mathrm{Exp}(\mu_s)$, đến $\sim$ Poisson($\lambda_s$) | Phân phối thời gian lưu trú; tỉ lệ thời gian theo phân phối dừng |
| Bounded Pareto | $F(x)=\frac{1-(L/x)^\alpha}{1-(L/H)^\alpha}$, $L\le x\le H$ | CCDF log-log, KS test |
| ON/OFF | N nguồn, ON/OFF $\sim$ Bounded Pareto, Poisson khi ON | Variance–time plot, hệ số Hurst (Poisson: 0,5) |
| Chung | Thời gian giữa hai lần đến của Poisson thuần nhất $\sim$ Exp | QQ-plot, KS test, index of dispersion |

In [ ]:
import importlib
import json
import math
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path('.').resolve()))
import arrival_models as am

plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})
SCENARIOS = ['nhpp', 'mmpp', 'pareto', 'onoff']
CFG = {s: importlib.import_module(f'scenarios.{s}').CONFIG for s in SCENARIOS}
DURATION, SEED = 1800, 1001
traces = {s: am.build_trace(CFG[s], DURATION, SEED) for s in SCENARIOS}
pd.DataFrame({s: {'arrival / run': len(t['events']), 'rps kỳ vọng (dài hạn)': round(t['expected_mean_rps'], 1),
                  'rps thiết kế (run này)': round(t['designed_mean_rps'], 1)} for s, t in traces.items()}).T

## 1. Tải thiết kế của 4 kịch bản (một run, seed 1001)
- **Đường đỏ:** tốc độ request kỳ vọng $E[\text{rps}](t)$ suy ra từ mô hình.
- **Cột xám:** số request do trace sinh ra trong mỗi 10s. Với kịch bản theo phiên, mỗi phiên được tính bằng số request của nó, đặt tại thời điểm phiên bắt đầu.

In [ ]:
def request_counts(trace, cfg, step=10):
    t = np.array([e['t'] for e in trace['events']])
    if cfg['model'] == 'onoff':
        w = np.ones(len(t))
    elif cfg['model'] == 'pareto':
        w = np.array([3 + e['size'] for e in trace['events']], float)
    else:
        w = np.array([1 + e['size'] for e in trace['events']], float)
    return np.histogram(t, np.arange(0, DURATION + 1, step), weights=w)[0] / step


fig, axes = plt.subplots(2, 2, figsize=(15, 8), sharex=True)
for ax, s in zip(axes.flat, SCENARIOS):
    tr = traces[s]
    exp_rps = np.array([r['expected_rps'] for r in tr['rate']])
    ax.bar(np.arange(0, DURATION, 10) / 60, request_counts(tr, CFG[s]), width=10 / 60, color='0.7', label='trace (10s)')
    ax.plot(np.arange(DURATION) / 60, exp_rps, 'r-', lw=1.2, label='E[rps](t) thiết kế')
    ax.set_title(f"{s.upper()} - trung bình {tr['designed_mean_rps']:.1f} rps")
    ax.set_ylabel('request/s')
    ax.legend(fontsize=8)
for ax in axes[-1]:
    ax.set_xlabel('phút')
fig.tight_layout()
plt.show()

## 2. Poisson thuần nhất: thời gian giữa hai lần đến ~ Exp
Kịch bản `pareto` có phiên đến theo Poisson thuần nhất (λ = 3 phiên/s). Với Poisson, thời gian giữa hai lần đến phải theo phân phối mũ: CV = 1, QQ-plot nằm trên đường chéo, và KS test không bác bỏ giả thuyết.

In [ ]:
def ks(sample, cdf):
    x = np.sort(np.asarray(sample))
    n = len(x)
    F = cdf(x)
    return max(np.max(np.arange(1, n + 1) / n - F), np.max(F - np.arange(n) / n)), 1.36 / math.sqrt(n)


lam = CFG['pareto']['session_rate']
ia = np.diff([e['t'] for e in traces['pareto']['events']])
d, crit = ks(ia, lambda x: 1 - np.exp(-lam * x))
q = np.linspace(0.005, 0.995, 200)
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.plot(-np.log(1 - q) / lam, np.quantile(ia, q), '.', ms=4)
ax.plot([0, ia.max()], [0, ia.max()], 'r--')
ax.set_xlabel('phân vị Exp(λ) lý thuyết')
ax.set_ylabel('phân vị thực nghiệm')
ax.set_title(f'CV = {ia.std() / ia.mean():.3f} | KS = {d:.4f} (ngưỡng 5%: {crit:.4f})')
plt.show()

## 3. NHPP: số lần đến khớp $\int \lambda(t)\,dt$
Sinh 30 trace với cùng pha, đếm số phiên mỗi phút rồi so với tích phân của λ(t) trên phút đó.

In [ ]:
cfg = CFG['nhpp']
fn = am.nhpp_rate(cfg, 0.0)
rmax = cfg['base'] + cfg['amp1'] + cfg['amp2']
reps = [am.nhpp_times(random.Random(k), fn, rmax, 0, DURATION) for k in range(30)]
edges = np.arange(0, DURATION + 1, 60)
emp = np.mean([np.histogram(r, edges)[0] for r in reps], axis=0)
theory = np.array([sum(fn(x + 0.5) for x in range(a, a + 60)) for a in edges[:-1]])
fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(edges[:-1] / 60, theory, 'r-', label='∫λ(t)dt mỗi phút')
ax.plot(edges[:-1] / 60, emp, 'o', ms=4, label='trung bình 30 trace')
ax.set_xlabel('phút')
ax.set_ylabel('phiên / phút')
ax.legend()
ax.set_title(f'Sai lệch tương đối lớn nhất: {np.max(np.abs(emp - theory) / theory):.1%}')
plt.show()

## 4. MMPP: thời gian lưu trú phân phối mũ, tỉ lệ thời gian theo phân phối dừng

In [ ]:
cfg = CFG['mmpp']
horizon = 3_000_000
path = am.ctmc_path(random.Random(3), cfg['states'], cfg['transitions'], cfg['initial'], horizon)
fig, axes = plt.subplots(1, 4, figsize=(17, 3.5))
rows = []
for ax, st in zip(axes, cfg['states']):
    soj = np.array([b - a for a, b, s in path if s == st][:-1])
    m = cfg['states'][st]['mean_sojourn']
    ax.hist(soj, bins=60, density=True, alpha=0.6)
    xs = np.linspace(0, soj.max(), 200)
    ax.plot(xs, np.exp(-xs / m) / m, 'r-')
    ax.set_title(f'{st}: mean {soj.mean():.1f}s (cấu hình {m:.0f}s)')
    rows.append({'trạng thái': st, 'tỉ lệ thời gian': soj.sum() / horizon, 'λ phiên/s': cfg['states'][st]['rate']})
order = list(cfg['states'])
states = [r['state'] for r in traces['mmpp']['rate']]
axes[3].plot(np.arange(DURATION) / 60, [order.index(s) for s in states], drawstyle='steps-post')
axes[3].set_yticks(range(len(order)), order)
axes[3].set_xlabel('phút')
axes[3].set_title('Trạng thái ẩn (seed 1001)')
fig.tight_layout()
plt.show()
pd.DataFrame(rows).round(3)

## 5. Bounded Pareto: đuôi nặng của khối lượng phiên
Trên thang log-log, CCDF $P(X > x)$ gần như là một đường thẳng có độ dốc $-\alpha$ (luật lũy thừa) và bị cắt tại H. Hầu hết phiên chỉ mua 1–3 sản phẩm, nhưng một số ít phiên mua hàng chục sản phẩm và chiếm phần lớn khối lượng checkout.

In [ ]:
p = CFG['pareto']['items']
rng = random.Random(5)
x = np.array([am.bounded_pareto(rng, p['alpha'], p['low'], p['high']) for _ in range(200_000)])
xs = np.logspace(0, np.log10(p['high']), 100)
cut = (p['low'] / p['high']) ** p['alpha']
theory = ((p['low'] / xs) ** p['alpha'] - cut) / (1 - cut)
emp = np.array([(x > v).mean() for v in xs])
items = np.ceil(x)
top = np.sort(items)[::-1]
share = top[:len(top) // 10].sum() / top.sum()
d, crit = ks(x[:20000], lambda v: (1 - (p['low'] / v) ** p['alpha']) / (1 - cut))
fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.loglog(xs, emp, '.', label='thực nghiệm')
ax.loglog(xs, np.maximum(theory, 1e-6), 'r-', label='lý thuyết')
ax.set_xlabel('x (số sản phẩm)')
ax.set_ylabel('P(X > x)')
ax.legend()
ax.set_title(f'KS = {d:.4f} (ngưỡng {crit:.4f}) | 10% phiên lớn nhất chiếm {share:.0%} số sản phẩm')
plt.show()
print(f"E[X] = {x.mean():.2f} (lý thuyết {am.bounded_pareto_mean(p['alpha'], p['low'], p['high']):.2f}); "
      f"E[ceil X] = {items.mean():.2f} (lý thuyết {am.bounded_pareto_ceil_mean(p['alpha'], p['low'], p['high']):.2f})")

## 6. ON/OFF: tính tự tương tự (Hurst)

**Variance–time plot:** gộp chuỗi số request mỗi giây thành các khối m giây, rồi tính phương sai của trung bình khối.
- Quá trình không có tương quan dài hạn (Poisson): $\mathrm{Var} \propto m^{-1}$, nên $H = 0{,}5$.
- Tải tự tương tự: phương sai giảm chậm hơn ($\propto m^{2H-2}$, $H > 0{,}5$), nghĩa là tải vẫn dồn cục khi nhìn ở thang thời gian lớn.

Theo lý thuyết, ON/OFF với Pareto cho $H = (3-\alpha_{\min})/2 = 0{,}85$. Vì thời gian ON/OFF bị chặn trên (300 s và 600 s), tính tự tương tự chỉ đúng tới khoảng vài phút, nên H ước lượng được sẽ thấp hơn giá trị lý thuyết.

In [ ]:
cfg = CFG['onoff']
long = am.build_trace(cfg, 6 * 3600, 7)
x = np.histogram([e['t'] for e in long['events']], np.arange(0, 6 * 3600 + 1))[0].astype(float)
pois = np.random.default_rng(0).poisson(x.mean(), len(x)).astype(float)
ms = np.array([2 ** k for k in range(0, 11)])


def vt(series):
    return np.array([series[:len(series) // m * m].reshape(-1, m).mean(1).var() for m in ms])


def hurst(v):
    return 1 + np.polyfit(np.log(ms[1:]), np.log(v[1:]), 1)[0] / 2


v_on, v_po = vt(x), vt(pois)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].loglog(ms, v_on / v_on[0], 'o-', label=f'ON/OFF  H = {hurst(v_on):.2f}')
axes[0].loglog(ms, v_po / v_po[0], 's-', label=f'Poisson H = {hurst(v_po):.2f}')
axes[0].loglog(ms, 1 / ms, 'k--', lw=0.8, label='độ dốc -1 (H = 0,5)')
axes[0].set_xlabel('kích thước khối m (giây)')
axes[0].set_ylabel('Var chuẩn hóa')
axes[0].legend()
axes[0].set_title(f'Variance-time plot (lý thuyết H = {am.theoretical_hurst(cfg):.2f})')
for m, c in [(1, '0.6'), (10, 'C0'), (60, 'C3')]:
    seg = x[:3600][: 3600 // m * m].reshape(-1, m).mean(1)
    axes[1].plot(np.arange(len(seg)) * m / 60, seg, color=c, lw=1, label=f'trung bình {m}s')
axes[1].set_xlabel('phút')
axes[1].set_ylabel('request/s')
axes[1].legend()
axes[1].set_title('ON/OFF: vẫn dồn cục khi gộp thang thời gian lớn')
fig.tight_layout()
plt.show()

## 7. Độ phân tán: index of dispersion
$\mathrm{IDC} = \mathrm{Var}(N)/E[N]$, với N là số request mỗi 10s.
- Poisson thuần nhất cho IDC ≈ 1.
- IDC > 1 nghĩa là tải dồn cục, mỗi kịch bản vì một nguyên nhân khác nhau:
  - **NHPP:** λ(t) thay đổi theo chu kỳ, nên phương sai gồm cả biến thiên của λ;
  - **MMPP:** chuyển giữa các trạng thái có λ rất khác nhau;
  - **Pareto:** phiên đến theo Poisson nhưng mỗi phiên mang số request có đuôi nặng (Poisson phức hợp: IDC ≈ E[S²]/E[S]);
  - **ON/OFF:** nguồn bật/tắt với thời gian đuôi nặng.
- Với tham chiếu: các kịch bản Locust hiện tại (mô hình đóng) có IDC < 1 khi tải ổn định, vì thời gian nghỉ theo phân phối đều.

In [ ]:
rows = []
for s in SCENARIOS:
    c = request_counts(traces[s], CFG[s]) * 10
    rows.append({'kịch bản': s, 'E[N] / 10s': c.mean(), 'Var[N]': c.var(), 'IDC': c.var() / c.mean()})
ref = np.random.default_rng(1).poisson(200, 180)
rows.append({'kịch bản': 'Poisson tham chiếu', 'E[N] / 10s': ref.mean(), 'Var[N]': ref.var(), 'IDC': ref.var() / ref.mean()})
pd.DataFrame(rows).set_index('kịch bản').round(2)

## 8. Dữ liệu thật: tải thiết kế so với tải thực nhận
Khi đã có run trong `results/`, so sánh `expected_rps` (thiết kế) với số request vào frontend đếm từ span Jaeger. Với mô hình mở, hai đường phải gần nhau **kể cả khi cụm quá tải**.

Nếu tải thực nhận thấp hơn rõ rệt, kiểm tra hai chỗ:
- `driver_report.json`: driver có phải bỏ arrival vì loadgen quá tải không;
- `jaeger_coverage` trong notebook tiền xử lý: Jaeger có bị rơi span không.

In [ ]:
run_metas = sorted(Path('results').glob('*/run_*/meta.json'))
if not run_metas:
    print('Chưa có run thật trong results/ - chạy: bash math-load-testing/run-scenario.sh <scenario> <runs>')
else:
    rows = []
    for mp in run_metas:
        d = mp.parent
        meta = json.loads(mp.read_text())
        rep = json.loads((d / 'driver_report.json').read_text()) if (d / 'driver_report.json').exists() else {}
        des = pd.read_csv(d / 'designed_rate.csv')
        svc = pd.read_csv(d / 'service_metrics.csv') if (d / 'service_metrics.csv').exists() else pd.DataFrame()
        fe = svc[svc.service == 'frontend'] if len(svc) else svc
        measured = fe.request_count.sum() / (meta['end'] - meta['start']) if len(fe) else np.nan
        rows.append({'run': f"{meta['scenario']}/{meta['run_id']}", 'seed': meta.get('seed'),
                     'rps thiết kế': des.expected_rps.mean(), 'rps frontend (Jaeger)': measured,
                     'thực / thiết kế': measured / des.expected_rps.mean(),
                     'arrival bị bỏ': rep.get('dropped'), 'max in-flight': rep.get('max_in_flight')})
    print(pd.DataFrame(rows).round(3).to_string())
    d = run_metas[0].parent
    meta = json.loads(run_metas[0].read_text())
    des = pd.read_csv(d / 'designed_rate.csv')
    fe = pd.read_csv(d / 'service_metrics.csv').query("service == 'frontend'")
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(des.t / 60, des.expected_rps, 'r-', lw=1, label='thiết kế')
    ax.plot((fe.timestamp - meta['start']) / 60, fe.request_count / 10, 'k-', lw=1, label='frontend thực nhận (Jaeger)')
    ax.set_xlabel('phút')
    ax.set_ylabel('request/s')
    ax.legend()
    ax.set_title(f"{meta['scenario']}/{meta['run_id']}")
    plt.show()